<a href="https://colab.research.google.com/github/gd-Sahat/ClockBiasPINN/blob/main/tsf_mamba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --quiet torch mamba-ssm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 124.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.4 MB/

In [ ]:
!apt-get update -qq && apt-get install -qq libomp-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libomp5-14:amd64.
(Reading database ... 126308 files and directories currently installed.)
Preparing to unpack .../libomp5-14_1%3a14.0.0-1ubuntu1.1_amd64.deb ...
Unpacking libomp5-14:amd64 (1:14.0.0-1ubuntu1.1) ...
Selecting previously unselected package libomp-14-dev.
Preparing to unpack .../libomp-14-dev_1%3a14.0.0-1ubuntu1.1_amd64.deb ...
Unpacking libomp-14-dev (1:14.0.0-1ubuntu1.1) ...
Selecting previously unselected package libomp-dev:amd64.
Preparing to unpack .../libomp-dev_1%3a14.0-55~exp2_amd64.deb ...
Unpacking libomp-dev:amd64 (1:14.0-55~exp2) ...
Setting up libomp5-14:amd64 (1:14.0.0-1ubuntu1.1) ...
Setting up libomp-14-dev (1:14.0.0-1ubuntu1.1) ...
Setting up libomp-dev:amd64 (1:14.0-55~exp2) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) 

In [ ]:
!pip install mamba-ssm --no-build-isolation

  Using cached mamba_ssm-2.2.4.tar.gz (91 kB)
  Preparing metadata (pyproject.toml) ... done
  Using cached ninja-1.11.1.4-py3-none-manylinux_2_12_x86_64.manylinux2010_x86_64.whl.metadata (5.0 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cuspa

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from mamba_ssm import Mamba
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from numpy.lib.stride_tricks import sliding_window_view

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
DATA_PATH = '/content/drive/MyDrive/CLOCKBIASPINN/clkbias_diff.gz'

In [ ]:
df = pd.read_csv(DATA_PATH, compression='gzip')

In [ ]:
# If timestamp is not datetime, convert it
if not np.issubdtype(df['timestamp'].dtype, np.datetime64):
    df['timestamp'] = pd.to_datetime(df['timestamp'])
# Create time_ns feature directly from datetime (nanoseconds since epoch)
df['time_ns'] = df['timestamp'].astype(np.int64)

# Encode satellite ID globally using factorize
df['id_enc'] = pd.factorize(df['id'])[0]

# Prepare raw features
raw_features = df[['id_enc','bias_interp_ns', 'time_ns']].values.astype(np.float32)

# Prepare and standardize targets
targets = df['bias_diff_ns'].values.astype(np.float32).reshape(-1, 1)
scaler_y = StandardScaler()
targets_scaled = scaler_y.fit_transform(targets).ravel()

# Combine encoded id_enc with raw features
features = raw_features

# Standardize features
scaler = StandardScaler()
features = scaler.fit_transform(features)

In [ ]:
# Create sliding windows
WINDOW_SIZE = 32
X = sliding_window_view(features, WINDOW_SIZE, axis=0)[:-1]  # (N-W, W, feat_dim)
y = targets_scaled[WINDOW_SIZE:]

# Split train/val/test (80/10/10)
total = len(X)
train_end = int(0.8 * total)
val_end = int(0.9 * total)
X_train, X_val, X_test = X[:train_end], X[train_end:val_end], X[val_end:]
y_train, y_val, y_test = y[:train_end], y[train_end:val_end], y[val_end:]

In [ ]:
class ClockBiasDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
batch_size = 64
train_ds = ClockBiasDataset(X_train, y_train)
val_ds   = ClockBiasDataset(X_val, y_val)
test_ds  = ClockBiasDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size)
test_loader  = DataLoader(test_ds,  batch_size=batch_size)

/tmp/ipython-input-18-4088979609.py:3: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  self.X = torch.from_numpy(X).float()


In [ ]:
class ClockBiasModel(nn.Module):
    def __init__(self, input_dim, d_model=128, d_state=16, d_conv=4, expand=2, dropout=0.1):
        super().__init__()
        self.fc_in = nn.Linear(input_dim, d_model)
        self.dropout = nn.Dropout(dropout)
        self.mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.fc_out = nn.Linear(d_model, 1)
    def forward(self, x):
        h = self.fc_in(x)
        h = self.dropout(h)
        h = self.mamba(h)
        return self.fc_out(h[:, -1, :])

In [ ]:
input_dim = X_train.shape[2]
model = ClockBiasModel(input_dim).to(device)
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(Xb).squeeze()
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * Xb.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    total_loss = 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        preds = model(Xb).squeeze()
        total_loss += criterion(preds, yb).item() * Xb.size(0)
    return total_loss / len(loader.dataset)

In [ ]:
epochs = 50
for ep in range(1, epochs+1):
    train_loss = train_epoch(model, train_loader)
    val_loss   = eval_epoch(model, val_loader)
    print(f"Epoch {ep}/{epochs} - Train Loss: {train_loss:.6f} - Val Loss: {val_loss:.6f}")

KeyboardInterrupt: 